# 04 — Trajectory Analysis
Analyze GROMACS production trajectories.
Uses MDAnalysis, MDTraj, and GROMACS tools for comprehensive analysis.

## What you can do
- RMSD, RMSF, Rg profiles
- Secondary structure assignment (DSSP)
- Contact maps and contact probabilities
- Hydrogen bond analysis
- Principal component analysis (PCA)
- Lipid properties (for membrane simulations)

### Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
plt.style.use('seaborn-v0_8-darkgrid')
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────
TRAJECTORY = "data/my_protein_md/prod.xtc"
TOPOLOGY = "data/my_protein_prep/em.gro"     # Reference structure
TOPOLOGY_TOP = "data/my_protein_prep/topol.top"
OUTPUT_DIR = "data/my_protein_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/plots", exist_ok=True)

print(f"Trajectory: {TRAJECTORY}")
print(f"Topology: {TOPOLOGY}")
print(f"Analysis output: {OUTPUT_DIR}")

## Load trajectory with MDAnalysis

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align, contacts, hydrogenbonds, pca

# Load universe
if os.path.exists(TRAJECTORY) and os.path.exists(TOPOLOGY):
    u = mda.Universe(TOPOLOGY, TRAJECTORY)
    print(f"Universe loaded:")
    print(f"  Frames: {u.trajectory.n_frames}")
    print(f"  Atoms: {u.atoms.n_atoms}")
    print(f"  Residues: {u.residues.n_residues}")
    print(f"  Time range: {u.trajectory[0].time:.1f} — {u.trajectory[-1].time:.1f} ps")
    
    # Select protein
    protein = u.select_atoms("protein")
    print(f"  Protein atoms: {protein.n_atoms}")
    ca = u.select_atoms("name CA")
    print(f"  C-alpha atoms: {ca.n_atoms}")
else:
    print(f"Trajectory or topology not found.")
    print("Run with an existing XTC+PDB pair for testing:")
    print(f"  u = mda.Universe('topology.pdb', 'samples.xtc')")

## 1. RMSD and RMSF

In [ ]:
if 'u' in locals() and u.atoms.n_atoms > 0:
    protein = u.select_atoms("protein")
    ca = u.select_atoms("name CA")
    
    # RMSD
    R = rms.RMSD(u, u, select="name CA", groupselections=["backbone"])
    R.run()
    rmsd_df = pd.DataFrame(R.results.rmsd[:, 1:], columns=['time_ps', 'rmsd_ca', 'rmsd_bb'])
    rmsd_df['time_ns'] = rmsd_df['time_ps'] / 1000
    rmsd_df['rmsd_ca_a'] = rmsd_df['rmsd_ca'] * 10
    rmsd_df['rmsd_bb_a'] = rmsd_df['rmsd_bb'] * 10
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    ax1.plot(rmsd_df['time_ns'], rmsd_df['rmsd_ca_a'], 'steelblue', lw=0.5, label='Cα')
    ax1.plot(rmsd_df['time_ns'], rmsd_df['rmsd_bb_a'], 'coral', lw=0.5, alpha=0.7, label='Backbone')
    ax1.set_xlabel('Time (ns)')
    ax1.set_ylabel('RMSD (Å)')
    ax1.set_title('RMSD vs Reference')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # RMSF
    R = rms.RMSF(ca)
    import MDAnalysis as mda
from MDAnalysis.analysis import align as mda_align
mda_align.AlignTraj(u, u, select="name CA", in_memory=True).run()
R.run(u)
    
    residues = [r.resid for r in ca.residues]
    ax2.plot(residues, R.results.rmsf * 10, 'green', lw=0.8)
    ax2.set_xlabel('Residue number')
    ax2.set_ylabel('RMSF (Å)')
    ax2.set_title('Per-Residue Cα RMSF')
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots/rmsd_rmsf.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nMean CA RMSD: {rmsd_df['rmsd_ca_a'].mean():.2f} ± {rmsd_df['rmsd_ca_a'].std():.2f} Å")
    print(f"Mean RMSF: {R.results.rmsf.mean() * 10:.2f} Å")

## 2. Secondary Structure (DSSP)
Requires DSSP installed (`conda install -c conda-forge dssp` or mda.dssp)

In [ ]:
if 'u' in locals():
    try:
        from MDAnalysis.analysis.dssp import DSSP
        if u.trajectory.n_frames > 0:
            # Sample every 10th frame for speed
            stride = max(1, u.trajectory.n_frames // 50)
            frames = range(0, u.trajectory.n_frames, stride)
            dssp = DSSP(u, select="protein", )
            dssp.run()
            
            ss_data = dssp.results
            ss_codes = {0: 'Coil', 1: 'a-Helix', 2: 'b-Sheet', 3: 'Turn', 4: '3_10-Helix'}
            
            from collections import Counter
            # Convert DSSP results to array
            if isinstance(ss_data, dict):
                ss_arr = np.array(list(ss_data.values())).flatten()
            elif hasattr(ss_data, 'flatten'):
                ss_arr = np.array(ss_data).flatten()
            else:
                ss_arr = np.array(ss_data)
            ss_freq = Counter(int(x) if not isinstance(x, (int, np.integer)) else x for x in ss_arr)
            
            plt.figure(figsize=(8, 5))
            codes = sorted(ss_freq.keys())
            labels = [ss_codes.get(c, f'Code {c}') for c in codes]
            values = [ss_freq[c] / len(ss_data) for c in codes]
            bars = plt.bar(labels, values, alpha=0.7, color='mediumseagreen')
            plt.ylabel('Frequency')
            plt.title('Secondary Structure Content')
            for bar, v in zip(bars, values):
                plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                         f'{v:.1%}', ha='center', fontsize=9)
            plt.tight_layout()
            plt.savefig(f"{OUTPUT_DIR}/plots/secondary_structure.png", dpi=150)
            plt.show()
    except (ImportError, Exception) as e:
        print(f"DSSP not available: {e}")
        print("Install with: conda install -c conda-forge dssp")

## 3. Contact Map

In [ ]:
# contact analysis uses scipy

if 'u' in locals() and protein.n_atoms > 0:
    # Compute contact map (CA-CA distance < 8 Å)
    print("Computing contact frequencies... (this may take a moment)")
    
    n_res = ca.n_atoms
    if n_res > 0 and n_res < 500:
        # Compute per-frame contact map for CA atoms
        from scipy.spatial.distance import pdist, squareform
        
        n_frames_sample = min(u.trajectory.n_frames, 100)
        frame_idxs = np.linspace(0, u.trajectory.n_frames - 1, n_frames_sample, dtype=int)
        contact_map = np.zeros((n_res, n_res))
        
        for idx in frame_idxs:
            u.trajectory[idx]
            coords = ca.positions  # in Å
            dists = squareform(pdist(coords))
            contact_map += (dists < 8.0).astype(float)
        contact_map /= len(frame_idxs)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        im1 = ax1.imshow(contact_map, cmap='viridis', vmin=0, vmax=1, 
                         interpolation='nearest', aspect='auto')
        ax1.set_xlabel('Residue')
        ax1.set_ylabel('Residue')
        ax1.set_title('Contact Probability (CA < 8 Å)')
        plt.colorbar(im1, ax=ax1, label='Probability')
        
        # Contact difference from average (for identifying contacts)
        contact_diff = contact_map - contact_map.mean()
        im2 = ax2.imshow(contact_diff, cmap='RdBu_r', vmin=-0.5, vmax=0.5,
                         interpolation='nearest', aspect='auto')
        ax2.set_xlabel('Residue')
        ax2.set_ylabel('Residue')
        ax2.set_title('Contact Probability (mean-centered)')
        plt.colorbar(im2, ax=ax2, label='Deviation')
        
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/plots/contact_map.png", dpi=150)
        plt.show()
    else:
        print(f"Skipping contact map ({n_res} CA atoms — too many for notebook)")

## 4. Hydrogen Bonds

In [ ]:
if 'u' in locals() and protein.n_atoms > 0:
    try:
        hb = hydrogenbonds.HydrogenBondAnalysis(
            u, "protein", "protein",
            ,
            update_selections=False
        )
        hb.run()
        
        print(f"\nHydrogen bond statistics:")
        print(f"  Mean intra-protein H-bonds: {hb.count_by_time().mean():.1f}")
        print(f"  Max: {hb.count_by_time().max()}")
        print(f"  Min: {hb.count_by_time().min()}")
        
        # Plot hydrogen bond count over time
        hb_counts = hb.count_by_time()
        plt.figure(figsize=(12, 3))
        plt.plot(u.trajectory.time / 1000, hb_counts, 'teal', lw=0.5)
        plt.xlabel('Time (ns)')
        plt.ylabel('H-bond count')
        plt.title('Intra-protein Hydrogen Bonds')
        plt.grid(alpha=0.3)
        plt.savefig(f"{OUTPUT_DIR}/plots/hbonds.png", dpi=120)
        plt.show()
    except Exception as e:
        print(f"Hydrogen bond analysis error: {e}")

## 5. Principal Component Analysis (PCA)
Identify dominant motions in the trajectory.

In [ ]:
if 'u' in locals() and ca.n_atoms > 20:
    print("Running PCA on Cα atoms...")
    pc = pca.PCA(u, select="name CA", align=True, mean=None)
    pc.run()
    
    variance = pc.results.variance / pc.results.variance.sum()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Explained variance
    ax1.bar(range(1, min(11, len(variance)+1)), 
            variance[:min(10, len(variance))] * 100, alpha=0.7,
            color='steelblue')
    ax1.set_xlabel('Principal Component')
    ax1.set_ylabel('Variance explained (%)')
    ax1.set_title('PCA — Variance Explained')
    ax1.grid(alpha=0.3)
    
    # Projection on PC1-PC2
    transformed = pc.transform(ca, n_components=2)
    sc = ax2.scatter(transformed[:, 0], transformed[:, 1], 
                     c=range(len(transformed)), cmap='viridis',
                     s=2, alpha=0.6)
    ax2.set_xlabel('PC1')
    ax2.set_ylabel('PC2')
    ax2.set_title('Trajectory Projection (time-colored)')
    plt.colorbar(sc, ax=ax2, label='Frame')
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots/pca.png", dpi=150)
    plt.show()
    
    print(f"PC1: {variance[0]*100:.1f}% of variance")
    print(f"PC2: {variance[1]*100:.1f}% of variance")

## 6. Lipid Analysis (for Membrane Simulations)
Analyze lipid bilayer properties: area per lipid, membrane thickness, order parameters.

In [ ]:
# MDAnalysis provides lipid analysis tools
# Run only if this is a membrane simulation

LIPID_SIMULATION = False  # Set to True if membrane

if LIPID_SIMULATION:
    try:
        from MDAnalysis.analysis.leaflet import LeafletFinder
        from MDAnalysis.analysis.area_per_lipid import AreaPerLipid
        
        # Find leaflets
        heads = u.select_atoms("name P*")
        leaflets = LeafletFinder(u, heads)
        print(f"Found {len(leaflets.groups)} leaflets")
        
        # Area per lipid
        apl = AreaPerLipid(u, tail_selection="name C* ").run()
        print(f"Mean APL: {apl.results.apl:.2f} Å²")
    except ImportError:
        print("MDAnalysis lipid tools not available.")
else:
    print("Lipid analysis skipped (not a membrane simulation).")

## 7. Free Energy Landscape (PC1 vs PC2)
Construct a 2D free energy surface from PCA projections.

In [ ]:
if 'transformed' in locals():
    from scipy.stats import gaussian_kde
    
    # Compute 2D density
    x = transformed[:, 0]
    y = transformed[:, 1]
    
    kernel = gaussian_kde([x, y])
    xi, yi = np.meshgrid(
        np.linspace(x.min(), x.max(), 50),
        np.linspace(y.min(), y.max(), 50)
    )
    zi = kernel(np.vstack([xi.ravel(), yi.ravel()]))
    zi = zi.reshape(xi.shape)
    
    # Convert to free energy (kJ/mol)
    kT = 2.577  # kJ/mol at 310 K
    fe = -kT * np.log(zi / zi.max())
    fe[fe > 15 * kT] = 15 * kT  # Cap at 15 kT
    
    plt.figure(figsize=(8, 6))
    contour = plt.contourf(xi, yi, fe, levels=20, cmap='hot_r')
    plt.colorbar(contour, label='Free energy (kJ/mol)')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title('Free Energy Landscape')
    plt.grid(alpha=0.1)
    plt.savefig(f"{OUTPUT_DIR}/plots/free_energy_landscape.png", dpi=150)
    plt.show()

## Summary

All plots saved to `{OUTPUT_DIR}/plots/`.

### Key files produced:
- `rmsd_rmsf.png` — structural stability
- `contact_map.png` — residue contact frequencies
- `pca.png` — dominant motions
- `free_energy_landscape.png` — conformational sampling
- `hbonds.png` — hydrogen bond network

### Export data:
```python
# Save analysis data for further processing
rmsd_df.to_csv(f"{OUTPUT_DIR}/rmsd.csv", index=False)
np.save(f"{OUTPUT_DIR}/contact_map.npy", contact_map)
```